<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/601088_%E0%B8%81%E0%B8%A4%E0%B8%95%E0%B8%B4%E0%B8%98%E0%B8%B5_%E0%B8%89%E0%B8%B2%E0%B8%A2%E0%B9%81%E0%B8%AA%E0%B8%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/fd0900c0.jpg
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/b17ced93.jpg
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/5157729e.jpg
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/9b25615e.jpg
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/1df1da60.jpg
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test/43c5b7da.jpg
/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recog

# Setup

In [ ]:
# ใช้ PyTorch + torchvision ที่มีใน Kaggle
print('Setup complete')


Setup complete


# Import and Config

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

IMG_SIZE = 320
BATCH_SIZE = 32
EPOCHS = 6
LR = 1e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)


DEVICE = cpu


# Load Data

In [ ]:
def find_file(filename: str):
    matches = list(INPUT_ROOT.rglob(filename))
    return matches[0] if matches else None

def find_named_dir(dirname: str):
    matches = [p for p in INPUT_ROOT.rglob('*') if p.is_dir() and p.name == dirname]
    return matches[0] if matches else None

train_csv_path = find_file('train.csv')
sample_sub_path = find_file('sample_submission.csv')
train_dir = find_named_dir('train')
test_dir = find_named_dir('test')

print('train_csv_path =', train_csv_path)
print('sample_sub_path =', sample_sub_path)
print('train_dir =', train_dir)
print('test_dir =', test_dir)

if train_csv_path is None or sample_sub_path is None or train_dir is None or test_dir is None:
    raise FileNotFoundError('train.csv / sample_submission.csv / train / test not found')

train_df = pd.read_csv(train_csv_path)
sample_sub = pd.read_csv(sample_sub_path)

# เลือกโฟลเดอร์ที่มีไฟล์ภาพจริง
if not list(train_dir.glob('*.jpg')):
    nested = [p for p in train_dir.rglob('*') if p.is_dir() and list(p.glob('*.jpg'))]
    if nested:
        train_dir = nested[0]

if not list(test_dir.glob('*.jpg')):
    nested = [p for p in test_dir.rglob('*') if p.is_dir() and list(p.glob('*.jpg'))]
    if nested:
        test_dir = nested[0]

print('resolved train_dir =', train_dir)
print('resolved test_dir =', test_dir)

display(train_df.head())
display(sample_sub.head())
print(train_df['class'].value_counts())


train_csv_path = /kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv
sample_sub_path = /kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv
train_dir = /kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train
test_dir = /kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test
resolved train_dir = /kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train
resolved test_dir = /kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test


,image_name,class
0,ChokChai4_img_13-7956791_100-6031267_a187-2159...,0
1,ChokChai4_img_13-7961753_100-6031881_a185-9785...,0
2,ChokChai4_img_13-7969811_100-5906061_a180-5812...,0
3,ChokChai4_img_13-7970811_100-5906071_a180-5812...,0
4,ChokChai4_img_13-7971811_100-5906081_a180-5812...,0


,id,answer
0,e4b420b0,0.0
1,23efa479,0.0
2,1f0f2402,0.0
3,8a60480c,NaN
4,11f20127,NaN


class
0    1520
1    1433
Name: count, dtype: int64


# Prepare Paths

In [ ]:
train_df = train_df.copy()
train_df['image_path'] = train_df['image_name'].apply(lambda x: str(train_dir / x))

missing_train = (~train_df['image_path'].map(lambda x: Path(x).exists())).sum()
print('missing_train =', missing_train)
if missing_train > 0:
    raise FileNotFoundError('Some training images are missing')

test_images = sorted(test_dir.glob('*.jpg'))
if not test_images:
    test_images = sorted(test_dir.rglob('*.jpg'))

test_df = pd.DataFrame({
    'image_path': [str(p) for p in test_images],
    'id': [p.stem for p in test_images]
})

print('num test images =', len(test_df))
display(test_df.head())


missing_train = 0
num test images = 1550


,image_path,id
0,/kaggle/input/competitions/super-ai-engineer-s...,00162f19
1,/kaggle/input/competitions/super-ai-engineer-s...,004c4789
2,/kaggle/input/competitions/super-ai-engineer-s...,0059b42f
3,/kaggle/input/competitions/super-ai-engineer-s...,005f930c
4,/kaggle/input/competitions/super-ai-engineer-s...,009095e8


# Split Data

In [ ]:
train_part, valid_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df['class']
)

print('train_part =', train_part.shape)
print('valid_part =', valid_part.shape)
print(train_part['class'].value_counts(normalize=True))
print(valid_part['class'].value_counts(normalize=True))


train_part = (2362, 3)
valid_part = (591, 3)
class
0    0.514818
1    0.485182
Name: proportion, dtype: float64
class
0    0.514382
1    0.485618
Name: proportion, dtype: float64


# Dataset and Transforms

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

tta_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class HouseDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        image = self.transform(image)

        if self.is_test:
            return image, row['id']

        label = torch.tensor(float(row['class']), dtype=torch.float32)
        return image, label

train_ds = HouseDataset(train_part, train_tfms, is_test=False)
valid_ds = HouseDataset(valid_part, valid_tfms, is_test=False)
test_ds = HouseDataset(test_df, tta_tfms, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# Model

In [ ]:
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 1)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Model ready')


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 295MB/s]

Model ready


# Train and Validate

In [ ]:
def run_epoch(model, loader, train_mode=True):
    if train_mode:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_probs = []
    all_targets = []

    for batch in loader:
        images, targets = batch
        images = images.to(DEVICE)
        targets = targets.to(DEVICE).view(-1, 1)

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, targets)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        all_probs.extend(probs.tolist())
        all_targets.extend(targets.detach().cpu().numpy().ravel().tolist())
        total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_probs), np.array(all_targets)

best_acc = -1
best_thr = 0.5
best_state = None

for epoch in range(EPOCHS):
    train_loss, _, _ = run_epoch(model, train_loader, train_mode=True)
    valid_loss, valid_probs, valid_targets = run_epoch(model, valid_loader, train_mode=False)

    epoch_best_acc = -1
    epoch_best_thr = 0.5
    for thr in np.arange(0.2, 0.81, 0.02):
        pred = (valid_probs >= thr).astype(int)
        acc = accuracy_score(valid_targets.astype(int), pred)
        if acc > epoch_best_acc:
            epoch_best_acc = acc
            epoch_best_thr = float(thr)

    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.5f} | valid_loss={valid_loss:.5f} | valid_acc={epoch_best_acc:.5f} | thr={epoch_best_thr:.2f}')

    if epoch_best_acc > best_acc:
        best_acc = epoch_best_acc
        best_thr = epoch_best_thr
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    scheduler.step()

print('best_acc =', best_acc)
print('best_thr =', best_thr)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/6 | train_loss=0.38224 | valid_loss=0.15760 | valid_acc=0.93739 | thr=0.34


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 2/6 | train_loss=0.15937 | valid_loss=0.11597 | valid_acc=0.96616 | thr=0.50


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Epoch 4/6 | train_loss=0.07836 | valid_loss=0.10947 | valid_acc=0.96447 | thr=0.38


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 5/6 | train_loss=0.05955 | valid_loss=0.11686 | valid_acc=0.96616 | thr=0.56


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 6/6 | train_loss=0.05833 | valid_loss=0.11459 | valid_acc=0.96785 | thr=0.74
best_acc = 0.9678510998307953
best_thr = 0.7399999999999998


# Refit

In [ ]:
# ใช้ model ที่ดีที่สุดจาก validation ก่อน ถ้าจะดันเพิ่มค่อย refit ด้วย train ทั้งหมด
model.load_state_dict(best_state)
model.eval()
print('Loaded best validation model')


Loaded best validation model


# Predict and Submission

In [ ]:
all_ids = []
all_probs = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        all_probs.extend(probs.tolist())
        all_ids.extend(list(ids))

pred_df = pd.DataFrame({'id': all_ids, 'prob': all_probs})
pred_df['answer'] = (pred_df['prob'] >= best_thr).astype(int)

submission = sample_sub[['id']].copy()
submission = submission.merge(pred_df[['id', 'answer']], on='id', how='left')

missing = submission['answer'].isna().sum()
print('missing predictions =', missing)
if missing > 0:
    raise ValueError('Some ids in sample submission were not matched with test images')

submission['answer'] = submission['answer'].astype(int)
submission.to_csv(OUTPUT_PATH, index=False)

print('Saved submission to', OUTPUT_PATH)
display(submission.head(15))
print(submission['answer'].value_counts())


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


missing predictions = 0
Saved submission to /kaggle/working/submission.csv


,id,answer
0,e4b420b0,0
1,23efa479,0
2,1f0f2402,0
3,8a60480c,0
4,11f20127,0
5,16173ced,0
6,9c05fea1,0
7,11f04229,0
8,c055fbb7,0
9,ec045077,0


answer
0    820
1    730
Name: count, dtype: int64
